In [1]:
!pip install keras_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 12.7 MB/s eta 0:00:00


In [11]:
import tensorflow as tf
import tf_keras as keras

# Load your saved model (.h5 or SavedModel format)
model = keras.models.load_model('model.h5')
print("✅ Model loaded successfully!")


✅ Model loaded successfully!


In [13]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [16]:
# Set your important paths
image_folder_path = '/content/drive/MyDrive/drone_dataset_new/train/drone/'           # <-- Change
bounding_boxes_file = '/content/drive/MyDrive/drone_dataset_new/train/drone/bounding_boxes.labels'  # <-- Change


In [18]:
import json

# First load the file normally
with open(bounding_boxes_file, 'r') as f:
    labels_content = json.load(f)

# Now check structure
if 'boundingBoxes' in labels_content:
    bounding_boxes_info = labels_content['boundingBoxes']
    print("✅ Bounding boxes extracted correctly!")
else:
    print("❌ boundingBoxes key not found!")


✅ Bounding boxes extracted correctly!


In [19]:
print(list(bounding_boxes_info.items())[:1])

[('0250.jpg.5p8r81fg.ingestion-54c4c64498-bpmm6.jpg', [{'label': 'drone\r', 'width': 3064, 'height': 1429, 'x': 1853, 'y': 167}])]


In [46]:
import numpy as np
import tensorflow as tf
import os
import random
from PIL import Image

IMG_SIZE = 96  # input size
GRID_SIZE = 12  # FOMO output size

# Generate segmentation maps
def generate_segmentation_map(bboxes, img_width, img_height, grid_size=GRID_SIZE):
    seg_map = np.zeros((grid_size, grid_size), dtype=np.uint8)

    for bbox in bboxes:
        x = bbox['x']
        y = bbox['y']
        width = bbox['width']
        height = bbox['height']

        # Find center of bounding box
        cx = x + width / 2
        cy = y + height / 2

        # Map to grid cell
        grid_x = int((cx / img_width) * grid_size)
        grid_y = int((cy / img_height) * grid_size)

        grid_x = np.clip(grid_x, 0, grid_size - 1)
        grid_y = np.clip(grid_y, 0, grid_size - 1)

        seg_map[grid_y, grid_x] = 1  # Class 1 (drone)

    return seg_map

# Prepare entries
data_entries = []
for image_filename, bboxes in bounding_boxes_info.items():
    full_path = os.path.join(image_folder_path, image_filename)

    data_entries.append((full_path, bboxes))

# Shuffle
random.shuffle(data_entries)

# Split: 90% train, 10% val
split_idx = int(0.8 * len(data_entries))
train_entries = data_entries[:split_idx]
val_entries = data_entries[split_idx:]

# Dataset loading function
def load_image_and_segmentation(entry):
    img_path, bboxes = entry

    # Load grayscale image
    img = Image.open(img_path).convert('L')
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img_width, img_height = img.size

    img = np.array(img) / 255.0
    img = np.expand_dims(img, axis=-1)  # Shape: (H, W, 1)

    seg_map = generate_segmentation_map(bboxes, img_width, img_height, grid_size=GRID_SIZE)

    # Convert seg_map to one-hot
    seg_map = tf.one_hot(seg_map, depth=2)  # (12, 12, 2)

    return img, seg_map

# Generators
def train_generator():
    for entry in train_entries:
        yield load_image_and_segmentation(entry)

def val_generator():
    for entry in val_entries:
        yield load_image_and_segmentation(entry)

# Output signature
output_signature = (
    tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 1), dtype=tf.float32),
    tf.TensorSpec(shape=(GRID_SIZE, GRID_SIZE, 2), dtype=tf.float32)
)

train_dataset = tf.data.Dataset.from_generator(train_generator, output_signature=output_signature).batch(32).repeat().prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_generator(val_generator, output_signature=output_signature).batch(32).repeat().prefetch(tf.data.AUTOTUNE)


print("✅ Segmentation-ready datasets are prepared!")


✅ Segmentation-ready datasets are prepared!


In [31]:
# Custom weighted crossentropy loss
def weighted_categorical_crossentropy(weights):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)
        loss = -tf.reduce_sum(weights * y_true * tf.math.log(y_pred), axis=-1)
        return tf.reduce_mean(loss)
    return loss

# FOMO typically weights object class higher
weights = tf.constant([1.0, 100.0])  # [background, object(drone)]

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss=weighted_categorical_crossentropy(weights),
    metrics=['accuracy']
)

print("✅ Model compiled with weighted crossentropy!")


✅ Model compiled with weighted crossentropy!


In [32]:
# Fine-tuning training
fine_tune_epochs = 60  # target final epoch

steps_per_epoch = len(train_entries) // 32
validation_steps = len(val_entries) // 32

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    initial_epoch=35,
    epochs=60,
    verbose=1
)



Epoch 36/60
41/41 [==============================] - 64s 1s/step - loss: 11.3382 - accuracy: 0.9930 - val_loss: 11.4002 - val_accuracy: 0.9929
Epoch 37/60
41/41 [==============================] - 63s 2s/step - loss: 11.3329 - accuracy: 0.9930 - val_loss: 11.4012 - val_accuracy: 0.9929
Epoch 38/60
41/41 [==============================] - 63s 2s/step - loss: 11.3417 - accuracy: 0.9930 - val_loss: 11.4015 - val_accuracy: 0.9929
Epoch 39/60
41/41 [==============================] - 52s 1s/step - loss: 11.3417 - accuracy: 0.9930 - val_loss: 11.4017 - val_accuracy: 0.9929
Epoch 40/60
41/41 [==============================] - 48s 1s/step - loss: 11.3417 - accuracy: 0.9930 - val_loss: 11.4022 - val_accuracy: 0.9929
Epoch 41/60
41/41 [==============================] - 60s 1s/step - loss: 11.3417 - accuracy: 0.9930 - val_loss: 11.4024 - val_accuracy: 0.9929
Epoch 42/60
41/41 [==============================] - 46s 1s/step - loss: 11.3417 - accuracy: 0.9930 - val_loss: 11.4027 - val_accuracy: 0.9929

In [36]:
def load_bounding_boxes(labels_file_path):
    with open(labels_file_path, 'r') as f:
        bounding_boxes_info = json.load(f)
    return bounding_boxes_info

In [37]:
test_bounding_boxes_info = load_bounding_boxes('/content/drive/MyDrive/drone_dataset_new/test/drone/bounding_boxes.labels')

import json




test_entries = list(test_bounding_boxes_info.items())

test_generator = lambda: load_images_and_labels(test_entries, path_to_test_images, input_shape)
test_dataset = tf.data.Dataset.from_generator(
    test_generator, output_signature=output_signature
).batch(32).prefetch(tf.data.AUTOTUNE)


In [38]:
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"✅ Test Loss: {test_loss:.4f}")
print(f"✅ Test Accuracy: {test_accuracy:.4f}")


UnknownError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
NameError: name 'load_images_and_labels' is not defined
Traceback (most recent call last):

  File "/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/dataset_ops.py", line 865, in get_iterator
    return self._iterators[iterator_id]
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^

KeyError: np.int64(0)


During handling of the above exception, another exception occurred:


Traceback (most recent call last):

  File "/usr/local/lib/python3.11/dist-packages/tensorflow/python/ops/script_ops.py", line 269, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "/usr/local/lib/python3.11/dist-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/from_generator_op.py", line 198, in generator_py

In [42]:
# Create test_entries filtering out non-image entries
test_entries = [
    (img_filename, boxes)
    for img_filename, boxes in test_bounding_boxes_info.items()
    if img_filename.endswith('.jpg') or img_filename.endswith('.png')
]


In [43]:
path_to_test_images = '/content/drive/MyDrive/drone_dataset_new/test/drone/'

test_generator = lambda: load_images_and_labels(test_entries, path_to_test_images, input_shape=(96, 96, 1))

test_dataset = tf.data.Dataset.from_generator(
    test_generator,
    output_signature=(
        tf.TensorSpec(shape=(96, 96, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(12, 12, 2), dtype=tf.float32)
    )
).batch(32).prefetch(tf.data.AUTOTUNE)


In [44]:
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"✅ Test Loss: {test_loss:.4f}")
print(f"✅ Test Accuracy: {test_accuracy:.4f}")


OverflowError: cannot convert float infinity to integer

In [45]:
print(f"Total test entries found: {len(test_entries)}")
for i, (img_filename, boxes) in enumerate(test_entries[:5]):
    print(f"{i+1}. {img_filename}, boxes: {boxes}")

Total test entries found: 0
